# DrillholeSet — composition root end-to-end

`DrillholeSet` bundles the collar + survey table plus N named interval tables into one object so you can call `db.validate()` / `db.desurvey()` / `db.to_omf(...)` instead of threading three or four DataFrames through every function.  No new logic — it delegates to the existing function-based API.

This notebook walks the GSWA sample through the whole flow.

In [1]:
import baselode.drill.data as drill
from baselode.drill import DrillholeSet

DATA = '../test/data/gswa'
collar = drill.load_collars(f'{DATA}/gswa_sample_collars.csv')
survey = drill.load_surveys(f'{DATA}/gswa_sample_survey.csv')
assays = drill.load_assays(f'{DATA}/gswa_sample_assays.csv')
geology = drill.load_geology(f'{DATA}/gswa_sample_geology.csv')

# 20-hole subset of holes with at least two survey stations
counts = survey['hole_id'].value_counts()
sample_holes = counts[counts >= 2].head(20).index.tolist()
collar_sub = collar[collar['hole_id'].isin(sample_holes)]
survey_sub = survey[survey['hole_id'].isin(sample_holes)]
assays_sub = assays[assays['hole_id'].isin(sample_holes)]
geology_sub = geology[geology['hole_id'].isin(sample_holes)]
len(collar_sub), len(survey_sub), len(assays_sub), len(geology_sub)

/Users/tam/Code/darkmine/darkmine-oss/baselode/.venv/lib/python3.12/site-packages/baselode/drill/data.py:222: DtypeWarning: Columns (0: simplifiedMethod, 1: simplifiedDigest) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(source, **kwargs)


(20, 14608, 997, 0)

## Build the set

`add_table` is chainable.  Each table is registered under a name + a free-form `kind` tag.

In [2]:
db = (
    DrillholeSet(collar_sub, survey_sub,
                 crs='EPSG:32750', project='gswa-subset')
      .add_table('assay', assays_sub)
      .add_table('geology', geology_sub, kind='litho')
)
db

<DrillholeSet holes=20 survey_rows=14608 tables=['assay', 'geology']>

## Validate the whole set in one call

In [3]:
report = db.validate()
report['summary']

{'error': 0, 'warning': 0, 'info': 34}

## Desurvey and cache the trace

First call runs the algorithm; subsequent calls with the same args return the cached frame.

In [4]:
traces = db.desurvey(step=5.0)
print(f'{len(traces)} trace samples')
print(f'second call returns same object: {db.desurvey(step=5.0) is traces}')
traces.head()

14614 trace samples
second call returns same object: True


,hole_id,md,easting,northing,elevation,azimuth,dip
0,72183ForrestaniaFFD163W4,0.0,0.000000,0.000000,430.000000,269.399994,-71.779999
1,72183ForrestaniaFFD163W4,5.0,-1.563171,-0.021827,434.749317,269.000000,-71.779999
2,72183ForrestaniaFFD163W4,10.0,-3.137333,-0.054840,439.494932,268.600006,-71.510002
3,72183ForrestaniaFFD163W4,15.0,-4.733719,-0.093855,444.233074,268.600006,-71.239998
4,72183ForrestaniaFFD163W4,20.0,-6.347044,-0.133284,448.965474,268.600006,-71.099998


## Export to OMF

GSWA collars only carry lat/lon; for the OMF write we build a projected collar table from the heads of the desurveyed traces.  In a real project you'd typically project the collars via `baselode.extent.Extent.to_crs` upstream.

In [5]:
projected_collar = (
    traces.sort_values('md').groupby('hole_id').first().reset_index()
          [['hole_id', 'easting', 'northing', 'elevation']]
)

db_projected = (
    DrillholeSet(projected_collar, survey_sub, project='gswa-subset')
      .add_table('assay', assays_sub)
)

from pathlib import Path
out_path = Path('/tmp/gswa-set.omf')
db_projected.to_omf(
    out_path,
    value_cols={'assay': [c for c in ['Cu_PPM', 'Au_PPM'] if c in assays_sub.columns]},
)
f'{out_path}: {out_path.stat().st_size / 1024:.1f} KB'

'/tmp/gswa-set.omf: 1140.6 KB'